# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their @id, name, and description
record_set_infos = []
for record_set in metadata.record_sets:
    name = getattr(record_set, 'name', 'N/A')
    desc = getattr(record_set, 'description', 'N/A')
    print(f"RecordSet @id: {record_set.id}\n  Name: {name}\n  Description: {desc}\n")
    record_set_infos.append({'id': record_set.id, 'name': name, 'description': desc})

# For each record set, list its fields and column ids
for record_set in metadata.record_sets:
    print(f"Fields for RecordSet {record_set.id}:")
    for field in record_set.fields:
        field_name = getattr(field, 'name', 'N/A')
        field_desc = getattr(field, 'description', 'N/A')
        print(f"  Field @id: {field.id}    Name: {field_name}    Description: {field_desc}")
        if hasattr(field, 'columns'):
            for col in field.columns:
                print(f"    Column @id: {col.id}    Name: {getattr(col, 'name', 'N/A')}")
    print('---')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 
Refer to the above overview to select record set and field `@id`s.

In [ ]:
# Extract data from each record set
record_sets = [r['id'] for r in record_set_infos]
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded RecordSet {record_set_id} with {len(records)} records.")
        else:
            print(f"No records found for RecordSet {record_set_id}.")
    except Exception as e:
        print(f"Could not load records for RecordSet {record_set_id}: {e}")

# Display columns for the main data record set (choose the first non-empty one)
main_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rsid
        break

if main_record_set_id:
    print(f"Columns in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
import numpy as np
# Choose numeric fields for EDA

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Try to auto-detect numeric fields
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields detected: {numeric_fields}")
    if not numeric_fields:
        # Try to convert likely numeric columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
        print(f"After conversion, numeric fields: {numeric_fields}")
    # Pick the first numeric field, or choose manually if needed
    numeric_field = numeric_fields[0] if numeric_fields else None

    if numeric_field is not None:
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to groupby another field (categorical)
        # Pick the first object dtype field that's not the numeric one
        cat_fields = df.select_dtypes(include=[object]).columns.tolist()
        group_field = None
        for cat in cat_fields:
            if cat != numeric_field:
                group_field = cat
                break
        if group_field and not filtered_df.empty and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for groupby.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


In this notebook, we used the `mlcroissant` library to load and explore the FAIR^2 Human Protein Mass Spectrometry dataset defined by the Croissant metadata schema. We inspected the available record sets, examined their fields, loaded the main data into a `pandas` DataFrame, performed basic filtering and normalization, and visualized value distributions. This approach can be easily extended to more specialized analyses as required for your bioscience or ML applications.
